<a href="https://colab.research.google.com/github/NourHassan5678/Assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task



## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## **Task Type: Ranking / Scoring**

Lane 4's question — *which visible pages under-capture clicks or engagement, and should a reviewer look at first?* — is a *"which ones first?"* question, which maps to **ranking / scoring** rather than classification or clustering.

A yes/no **"is this a problem page?"** label would throw away exactly the information a review queue needs: **which page is more worth a reviewer's time than the next one**. So the output is a **priority score per page**, sorted so a reviewer works from the top down until their review capacity for the cycle runs out.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / proxy: a position-tier-relative CTR gap

What I'm scoring isn't a single observed column — it's a **derived measurement**: how far below what's typical for pages in the *same position tier* a page's CTR sits. It's a gap/residual I calculate myself, not a copy of any FlyRank product decision (nothing like `health_score` goes near it), so it stays on the safe side of the observable-signals-only rule.

Where the label actually comes from, honestly:

- The **inputs** (`ctr`, `avg_position`, `position_tier`) are real observed measurements from the 90-day window.
- The **expectation** each page is compared against (its tier's median CTR) is a statistic *I calculate from the data itself* — not a threshold I invented from nowhere.
- The **gap** is therefore a defined comparison, not a future observed outcome. It says a page currently sits below its peers — it does not claim the page will improve. That distinction matters for a proper supervised label.

The lane guide allows "classification, if you define a leakage-safe label" as a further step. The dataset actually gives me one for free: `impressions_last_30d`/`clicks_last_30d` vs. `impressions_prev_30d`/`clicks_prev_30d`. Comparing CTR between those two already-elapsed windows is a genuine **observed outcome** — did the page's CTR move between two known-past periods — not a rule I made up and not a leak from the future, since both windows are already over at snapshot time. I sketch both versions below.

In [ ]:
!git clone https://github.com/NourHassan5678/Assignments.git
import os
os.chdir("Assignments")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same "visible" filter as the EDA: enough impressions to matter, and a
# position where clicks are realistically available. Also drop the
# avg_position == 0 sentinel rows (found during EDA) so they don't
# contaminate the top_3 tier's expected CTR.
visible = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
].copy()

# --- Primary target/proxy: tier-relative CTR gap ---
# Expected CTR = median CTR of pages in the SAME position tier.
visible["expected_ctr_for_tier"] = visible.groupby("position_tier")["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["expected_ctr_for_tier"]
# Negative ctr_gap = under-capturing clicks relative to its own tier = opportunity.

print("Sketch of the primary target column (ctr_gap), worst gaps first:")
print(visible[["content_id", "position_tier", "ctr", "expected_ctr_for_tier", "ctr_gap"]]
      .sort_values("ctr_gap").head(5))

# --- Stretch target: an OBSERVED outcome from the two real time windows ---
# Only defined where both windows have enough impressions to compute a CTR at all.
has_both_windows = (visible["impressions_prev_30d"] > 0) & (visible["impressions_last_30d"] > 0)
w = visible[has_both_windows].copy()
w["ctr_prev_30d"] = w["clicks_prev_30d"] / w["impressions_prev_30d"]
w["ctr_last_30d"] = w["clicks_last_30d"] / w["impressions_last_30d"]
w["ctr_improved"] = (w["ctr_last_30d"] > w["ctr_prev_30d"]).astype(int)

print(f"\nPages with both windows defined: {len(w):,} / {len(visible):,}")
print(f"Observed CTR-improved rate: {w['ctr_improved'].mean():.1%}")
print("\nSketch of the stretch target column (ctr_improved):")
print(w[["content_id", "ctr_prev_30d", "ctr_last_30d", "ctr_improved"]].head(5))

Cloning into 'Assignments'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 22), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 8.20 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Sketch of the primary target column (ctr_gap), worst gaps first:
                 content_id position_tier  ctr  expected_ctr_for_tier  ctr_gap
15257  content_bc8ec57819c2        page_1  0.0                   0.24    -0.24
27026  content_b5d6346b5ad5        page_1  0.0                   0.24    -0.24
21855  content_b23fa9e12c1d        page_1  0.0                   0.24    -0.24
3886   content_a2d394e98056        page_1  0.0                   0.24    -0.24
3871   content_54d8f9de6ec2        page_1  0.0                   0.24    -0.24

Pages with both windows defined: 11,909 / 12,023
Observed CTR-improved rate: 47.7%

Sketch of the stretch target col

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@50 — and the action it's built around

**Metric: Precision@50.** Of the top 50 pages the ranking puts first, how many are genuinely worth a reviewer's time? I'm anchoring on a top-K metric, not accuracy or AUC across the whole dataset, because of the action this output is actually for:

**Action.** A content editor or SEO reviewer works down the ranked queue from the top, opening each flagged page to check its title, meta description, and intent match — until their review capacity for that cycle runs out. That's realistically 20-50 pages, not all 12,023 visible candidates. A metric computed over the *entire* ranked list, like plain accuracy, would reward getting the bottom of a 12,000-page list right — pages a reviewer will never open. Precision@K is the metric that actually matches how the list gets used.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content item (page), scored within its fixed 90-day snapshot window.** Not per-client, not per-day — Lane 4 ranks individual pages against their tier peers.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 4's slice: visible pages only, sentinel avg_position == 0 rows dropped.
lane4_df = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
].copy()

print(f"Lane 4 slice: {len(lane4_df):,} rows out of {len(df):,} total pages")
print("One row = one content item (page), measured over its 90-day window.\n")

lane4_df[[
    "content_id", "client_id", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "engagement_rate", "scroll_rate"
]].head(8)

Lane 4 slice: 12,023 rows out of 30,000 total pages
One row = one content item (page), measured over its 90-day window.



,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,engagement_rate,scroll_rate
0,content_304f48230142,client_f369cb89fc,striking,10.6,3803,29,0.76,5.88,4.55
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,11751,58,0.49,1.28,3.45
5,content_d4084a4bc775,client_f369cb89fc,page_1,8.5,3970,1,0.03,0.00,25.00
9,content_c27558df2b0c,client_19581e27de,page_1,4.9,1240,2,0.16,0.00,0.00
10,content_d8ee6cc6d642,client_19581e27de,top_3,2.2,20919,324,1.55,6.75,9.55
12,content_42fb2cad9ecf,client_6208ef0f77,page_1,5.6,7228,127,1.76,3.43,2.61
16,content_78bd1d4a1d4d,client_6208ef0f77,page_1,8.9,13848,21,0.15,0.21,13.83
17,content_761a44afda12,client_19581e27de,page_1,7.3,9449,7,0.07,11.86,15.25


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML (or at least a statistically-fit baseline) beats a fixed rule here

Section 2's own code already shows why a single global cutoff doesn't work: `expected_ctr_for_tier` is computed *separately per tier*, and it isn't the same number across tiers — a `page_1` page's baseline (0.24 median CTR) is not the baseline a `striking` or `page_3_5` page should be judged against. A flat rule like `ctr < 0.5` applies one number to every page regardless of tier, so it necessarily treats naturally-lower-CTR tiers as "more broken" than they really are, and naturally-higher-CTR tiers as "fine" even when they're underperforming their own peers.

The stretch target in Section 2 makes a related point: only 47.7% of pages with both time windows defined showed CTR improve between the previous and most recent 30 days — close to a coin flip. That's not a clean, rule-friendly pattern where one if-statement reliably separates "about to get better" from "about to get worse"; it takes weighing several signals together, not one static cutoff.

None of this needs a complex model to start — the tier-median baseline in Section 2 is itself a form of letting the data set the threshold, rather than hand-picking one number for every page. But combining several signals (CTR gap, engagement rate, scroll rate, volume) into one learned score, instead of one hand-picked cutoff, is where a simple trained model earns its place over the baseline — which is what the later weeks of this track (baseline → model → validation) are for.